# Hosted Agent Session Log Stream — `@azure/ai-projects`

This notebook demonstrates how to stream hosted agent session logs using `project.agents.getSessionLogStream` with the `AIProjectClient`.

Sessions only work with Hosted Agents. Session and log stream operations are currently preview features.

It mirrors the [`sessionLogStream.ts`](./sessionLogStream.ts) sample and runs the **locally built** `@azure/ai-projects` from this repo.

## Prerequisites

1. **Build the package first** so `dist/` is current: `cd sdk/ai/ai-projects && pnpm build`
2. **tslab kernel** installed and registered (`npm install -g tslab` then `tslab install`); select the **TypeScript** (tslab) kernel.
3. **Launch VS Code / Jupyter from `sdk/ai/ai-projects/`** so Node resolves the local `@azure/ai-projects`.
4. **`az login`** completed so `DefaultAzureCredential` can authenticate.
5. **Environment variables**: `FOUNDRY_PROJECT_ENDPOINT`, `FOUNDRY_AGENT_CONTAINER_IMAGE`.

Run the cells in order (top to bottom); state is shared across cells.

In [1]:
// Imports and configuration
import type {
  AgentEndpointConfig,
  FixedRatioVersionSelectionRule,
  HostedAgentDefinition,
  ProtocolVersionRecord,
  VersionRefIndicator,
} from "@azure/ai-projects";
import { AIProjectClient } from "@azure/ai-projects";
import { DefaultAzureCredential } from "@azure/identity";
import { execFileSync } from "node:child_process";

const projectEndpoint = process.env["FOUNDRY_PROJECT_ENDPOINT"] ?? "<project endpoint>";
const image = process.env["FOUNDRY_AGENT_CONTAINER_IMAGE"] ?? "<agent image>";
const agentName = "MySessionLogStreamAgent";

console.log(`Agent: ${agentName}`);
console.log(`Image: ${image}`);

Agent: MySessionLogStreamAgent
Image: crjep6bl5hlacma.azurecr.io/crjep6bl5hlacma/responses-echo-agent:latest
Image: crjep6bl5hlacma.azurecr.io/crjep6bl5hlacma/responses-echo-agent:latest


In [2]:
// Minimal Server-Sent Events (SSE) frame parser used to read the log stream
interface SseFrame {
  event: string | undefined;
  data: string;
}

async function* iterSseFrames(
  stream: NodeJS.ReadableStream,
  maxEvents: number,
): AsyncGenerator<SseFrame> {
  let buffer = "";
  let eventCount = 0;

  try {
    for await (const chunk of stream as AsyncIterable<Buffer>) {
      buffer += chunk.toString("utf-8");

      while (buffer.includes("\n\n")) {
        const idx = buffer.indexOf("\n\n");
        const frame = buffer.slice(0, idx);
        buffer = buffer.slice(idx + 2);

        let eventName: string | undefined;
        const dataLines: string[] = [];
        for (const line of frame.split("\n")) {
          if (line.startsWith("event: ")) {
            eventName = line.slice(7);
          } else if (line.startsWith("data: ")) {
            dataLines.push(line.slice(6));
          }
        }

        if (dataLines.length > 0 || eventName) {
          yield { event: eventName, data: dataLines.join("\n") };
          eventCount++;
          if (eventCount >= maxEvents) return;
        }
      }
    }
  } finally {
    if ("destroy" in stream && typeof stream.destroy === "function") {
      stream.destroy();
    }
  }
}

In [3]:
// Create the AI Project client
// Annotated as `any` so tslab does not try to emit non-portable declarations
// referencing the deep `node_modules/openai` (pnpm junction) path.
const project: any = new AIProjectClient(projectEndpoint, new DefaultAzureCredential());

In [5]:
// Create a hosted agent version from a container image
console.log("Creating agent...");
const agent = await project.agents.createVersion(
  agentName,
  {
    kind: "hosted",
    cpu: "0.5",
    memory: "1Gi",
    container_configuration: { image: image } as AgentEndpointConfig,
    protocol_versions: [{ protocol: "responses", version: "1.0.0" } as ProtocolVersionRecord],
  } as HostedAgentDefinition,
  {
    metadata: { enableVnextExperience: "true" },
  },
);
console.log(`Agent created (name: ${agent.name}, version: ${agent.version})`);

Creating agent...
Agent created (name: MySessionLogStreamAgent, version: 2)


In [7]:
// Poll until the agent version is active
for (let attempt = 0; attempt < 60; attempt++) {
  await new Promise((resolve) => setTimeout(resolve, 3_000));
  const versionDetails = await project.agents.getVersion(agentName, agent.version);
  const status = versionDetails.status;
  console.log(`Agent version status: ${status} (attempt ${attempt + 1}/60)`);
  if (status === "active") break;
  if (status === "failed") {
    throw new Error(`Agent version provisioning failed: ${JSON.stringify(versionDetails)}`);
  }
  if (attempt === 59) {
    throw new Error("Timed out waiting for agent version to become active");
  }
}

Agent version status: active (attempt 1/60)


In [8]:
// Create a session
const versionIndicator: VersionRefIndicator = {
  type: "version_ref",
  agent_version: agent.version,
};
const session = await project.agents.createSession(agentName, versionIndicator);
console.log(`Session created (id: ${session.agent_session_id}, status: ${session.status})`);

Session created (id: 1f72ce3766f6967900UMDmCTuSXgiZCHDHYpsPe8jNWaeuJCxF, status: active)


In [ ]:
// Capture the current endpoint so cleanup can restore it, then configure the
// agent endpoint for the responses protocol.
const originalAgentEndpoint = (await project.agents.get(agentName)).agent_endpoint;

const endpointConfig: AgentEndpointConfig = {
  version_selector: {
    version_selection_rules: [
      {
        type: "FixedRatio",
        agent_version: agent.version,
        traffic_percentage: 100,
      } as FixedRatioVersionSelectionRule,
    ],
  },
  protocol_configuration: { responses: {} },
};

await project.agents.updateAgent(agentName, { agentEndpoint: endpointConfig });
console.log(`Agent endpoint configured for agent: ${agentName}`);

In [11]:
// Call the Responses API bound to the agent session
// Annotated as `any` so tslab does not emit a non-portable declaration
// referencing the deep `node_modules/openai` (pnpm junction) path.
const openAIClient: any = project.getOpenAIClient({
  azureConfig: { allowPreview: true, agentName: agentName },
});

console.log("Generating response...");
const response = await openAIClient.responses.create(
  {
    input: "Say hello in one short sentence.",
  },
  {
    body: { agent_session_id: session.agent_session_id },
  },
);
console.log(`Response output: ${response.output_text}`);

Generating response...
Response output: Echo: Say hello in one short sentence.


In [12]:
// Stream session logs
await new Promise((resolve) => setTimeout(resolve, 2_000));

console.log("Streaming session logs...");
const logStream = await project.agents.getSessionLogStream(
  agentName,
  agent.version,
  session.agent_session_id,
);

const streamLogs = async () => {
  if (logStream.readableStreamBody) {
    for await (const frame of iterSseFrames(logStream.readableStreamBody, 30)) {
      console.log(`SSE event: ${frame.event}`);
      console.log(`SSE data: ${frame.data}\n`);
      console.log("Frame received:", JSON.stringify(frame));
      console.log("-----");
    }
  }
};
await streamLogs();

Streaming session logs...
SSE event: log
SSE data: {"timestamp":"2026-08-11T23:49:20.26+00:00","session_id":"1f72ce3766f6967900UMDmCTuSXgiZCHDHYpsPe8jNWaeuJCxF","session_state":"Running","agent":"MySessionLogStreamAgent","generated_at":"2026-08-11T23:50:58.9575774+00:00","last_accessed":"2026-08-11T23:49:20.26+00:00"}

Frame received: {"event":"log","data":"{\"timestamp\":\"2026-08-11T23:49:20.26+00:00\",\"session_id\":\"1f72ce3766f6967900UMDmCTuSXgiZCHDHYpsPe8jNWaeuJCxF\",\"session_state\":\"Running\",\"agent\":\"MySessionLogStreamAgent\",\"generated_at\":\"2026-08-11T23:50:58.9575774+00:00\",\"last_accessed\":\"2026-08-11T23:49:20.26+00:00\"}"}
-----
SSE event: log
SSE data: {"timestamp":"2026-08-11T23:50:59.036130568Z","stream":"status","message":"Connecting to the container..."}

Frame received: {"event":"log","data":"{\"timestamp\":\"2026-08-11T23:50:59.036130568Z\",\"stream\":\"status\",\"message\":\"Connecting to the container...\"}"}
-----
SSE event: log
SSE data: {"timestamp":

In [ ]:
// Clean up resources
console.log("Cleaning up resources...");

try {
  await project.agents.deleteSession(agentName, session.agent_session_id);
  console.log(`Session deleted (id: ${session.agent_session_id})`);
} catch (e) {
  console.error("Failed to delete session:", e);
}
try {
  if (originalAgentEndpoint !== undefined) {
    await project.agents.updateAgent(agentName, { agentEndpoint: originalAgentEndpoint });
    console.log("Agent endpoint restored to previous configuration");
  }
} catch (e) {
  console.error("Failed to restore endpoint:", e);
}
try {
  await project.agents.deleteVersion(agentName, agent.version);
  console.log(`Agent version ${agent.version} deleted.`);
} catch (e) {
  console.error("Failed to delete agent version:", e);
}